In [1]:
from __future__ import annotations
import sys
from pathlib import Path
from pprint import pprint
from typing import TypedDict

from sqlalchemy.ext.asyncio import result

from src.rag import ( MultimodalRetriever, load_documents, prepare_rag_prompt)
import re
from collections import  Counter
from langgraph.graph import StateGraph, END
from langgraph.checkpoint.memory import MemorySaver

In [2]:
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"Project root: {PROJECT_ROOT}")

Project root: C:\agentic\Property-Rental-Management-System


In [3]:
#Shared agent state
class AgentState(TypedDict, total=False):
    """Shared state passed between every note in the orchestration graph"""

    message: str
    tenant_id: str
    unit_id: str
    intent: str
    rag_context: str
    response: str
    agent_used: str
    confidence: float

print("AgentState keys: ", list(AgentState.__annotations__.keys()))

AgentState keys:  ['message', 'tenant_id', 'unit_id', 'intent', 'rag_context', 'response', 'agent_used', 'confidence']


In [4]:
ASSET_DIR = PROJECT_ROOT / "notebooks" / "_generated_assets" / "06_agent_orchestration"
ASSET_DIR.mkdir(parents=True, exist_ok=True)

docs_content = {
   "tenant_policy.txt": (
         "Unit A1 lease: monthly rent R4500, due on the 1st. "
        "Pets  allowed with written approval. Quiet hours 22:00-07:00."
         "Rooftop garden access 08:00-20:00. Parking bay 12 assigned. "
  ),
    "payment_log.txt": (
        "Tenant Bennet Dyani (unit A1) paid July rent in full R4500 on 2026-07-03 - on time."
        "Tenant Itumeleng Bedesho (unit B2) rent R4500 due on 2026-07-01 - OVERDUE as of 2026-07-06."
        "Late fee policy: R100 after 5 days, R200 after 10 days, R300 after 15 days."
    ),
    "maintenance_log.txt": (
        "Unit A1: Leaking kitchen sink - approved, technician visit 2026-07-07 10:00. "
        "Unit B2: Bedroom light not working - pending landlord approval. Priority: medium. "
        "Unit C3:Front door lock jam - scheduled 2026-07-15."
    ),
    "financial_summary.txt": (
        "Q2 2026 rental income: R36,000 across 8 units. "
        "Vacancy rate: 12.5% (1 unit vacant). "
        "Projected Q3 income: R54,000 assuming full occupancy from August. "
        "Maintenance costs Q2: R3,100. Net operating income Q2: R25,300."
    ),
}

doc_paths = []
for filename, content in docs_content.items():
    path = ASSET_DIR / filename
    path.write_text(content, encoding="utf-8")
    doc_paths.append(path)

metadata_by_source = {
    str(ASSET_DIR / "tenant_policy.txt"): {"category": "lease"},
    str(ASSET_DIR / "payment_log.txt"): {"category": "payment"},
    str(ASSET_DIR / "maintenance_log.txt"): {"category": "maintenance"},
    str(ASSET_DIR / "financial_summary.txt"): {"category": "financial"}
}

documents = load_documents(doc_paths, metadata_by_source=metadata_by_source)

retriever = MultimodalRetriever()
retriever.add_documents(documents, chunk_size=20, chunk_overlap=5)

print(f"Corpus loaded: {len(documents)} documents,  {len(retriever.chunks)} chunks")



Corpus loaded: 4 documents,  11 chunks


In [5]:
_INTENT_KEYWORDS: dict[str, list[str]] = {
    "tenant_qa": [
        "lease", "policy", "rule", "pet", "parking", "amenity", "garden",
        "quiet", "hours", "allowed", "tenant", "unit", "property",
    ],
    "payment": [
        "rent", "payment", "paid", "overdue", "late", "fee", "invoice",
        "due", "balance", "charge", "receipt",
    ],
    "maintenance": [
        "repair", "fix", "broken", "leak", "maintenance", "request",
        "technician", "schedule", "hvac", "window", "tap", "issue",
    ],
    "forecast": [
        "forecast", "income", "revenue", "predict", "projection", "vacancy",
        "occupancy", "q2", "q3", "financial", "net", "operating",
    ],
}

def _tokenize(text: str) -> list[str]:
    return re.findall(r"[a-z0-9]+", text.lower())

def route_intent(state: AgentState) -> AgentState:
    """Classify the user message and write intent + confidence into state."""
    tokens = Counter(_tokenize(state["message"]))
    scores: dict[str, float] = {}

    for intent, keywords in _INTENT_KEYWORDS.items():
        scores[intent] = sum(tokens[kw] for kw in keywords)

    best_intent = max(scores, key=lambda k: scores[k])
    total = sum(scores.values()) or 1
    confidence = round(scores[best_intent] / total, 3)

    if scores[best_intent] == 0:
        best_intent = "unknown"
        confidence = 0.0

    return {"intent": best_intent, "confidence": confidence}

#smoke test

test_messages = [
    "What is the lease for unit A1?",
    "What is the rent for unit B2?",
    "What is the maintenance cost for unit C3?",
    "What is the Q3 income forecast?",
]

for msg in test_messages:
    result = route_intent({"message": msg})
    print(f"{msg!r:55s} -> intent: {result['intent']:10s} confidence: {result['confidence']:.2f}")

'What is the lease for unit A1?'                        -> intent: tenant_qa  confidence: 1.00
'What is the rent for unit B2?'                         -> intent: tenant_qa  confidence: 0.50
'What is the maintenance cost for unit C3?'             -> intent: tenant_qa  confidence: 0.50
'What is the Q3 income forecast?'                       -> intent: forecast   confidence: 1.00


In [6]:
def _mock_llm_call(prompt: str) -> str:
    """Mock LLM call that returns a response based on the prompt."""
    if "Context:" in prompt:
        context_part = prompt.split("Context:")[1].strip()
        return f"[Mock LLM] Based on the retrieved context: \n {context_part}"
    return "[Mock LLM ] No relevant context found."

In [7]:
def tenant_qa_agent(state: AgentState) -> AgentState:
    """Answer tenant questions about lease terms, policies, and amenities."""
    prompt = prepare_rag_prompt(state["message"], retriever, top_k=3, filters={"category": "lease"})
    return {
        "rag_context": retriever.build_context(state["message"], top_k=3, filters={"category": "lease"}),
        "response": _mock_llm_call(prompt),
        "agent_used": "TenantQAAgent",
    }

def payment_tracker_agent(state: AgentState) -> AgentState:
    """Report on rent payment status, overdue balances, and late fees."""
    prompt = prepare_rag_prompt(state["message"], retriever, top_k=3, filters={"category": "finance"})
    return {
        "rag_context": retriever.build_context(state["message"], top_k=3, filters={"category": "finance"}),
        "response": _mock_llm_call(prompt),
        "agent_used": "PaymentTrackerAgent",
    }

def maintenance_handler_agent(state: AgentState) -> AgentState:
    """Log maintenance requests and surface scheduled technician visits."""
    prompt = prepare_rag_prompt(state["message"], retriever, top_k=3, filters={"category": "maintenance"})
    return {
        "rag_context": retriever.build_context(state["message"], top_k=3, filters={"category": "maintenance"}),
        "response": _mock_llm_call(prompt),
        "agent_used": "MaintenanceHandlerAgent",
    }

def forecaster_agent(state: AgentState) -> AgentState:
    """Provide income projections, vacancy analysis, and financial summaries."""
    prompt = prepare_rag_prompt(state["message"], retriever, top_k=3, filters={"category": "forecast"})
    return {
        "rag_context": retriever.build_context(state["message"], top_k=3, filters={"category": "forecast"}),
        "response": _mock_llm_call(prompt),
        "agent_used": "ForecasterAgent",
    }

def unknown_agent(state: AgentState) -> AgentState:
    """Fallback for messages that don't match any known intent."""
    return {
        "rag_context": "",
        "response": (
            "I'm not sure how to help with that. "
            "You can ask about lease policies, rent payments, maintenance requests, "
            "or income forecasts."
        ),
        "agent_used": "UnknownAgent",
    }


print("Agent nodes defined: TenantQAAgent, PaymentTrackerAgent, MaintenanceHandlerAgent, ForecasterAgent, UnknownAgent")

Agent nodes defined: TenantQAAgent, PaymentTrackerAgent, MaintenanceHandlerAgent, ForecasterAgent, UnknownAgent


In [8]:
def _dispatch(state: AgentState) -> str:
    """Conditional edge: map intent to the appropriate agent."""
    return state.get("intent", "unknown")

builder = StateGraph(AgentState)

builder.add_node("router", route_intent)
builder.add_node("tenant_qa", tenant_qa_agent)
builder.add_node("payment", payment_tracker_agent)
builder.add_node("maintenance", maintenance_handler_agent)
builder.add_node("forecast", forecaster_agent)
builder.add_node("unknown", unknown_agent)

builder.set_entry_point("router")

builder.add_conditional_edges(
    "router",
    _dispatch,
    {
        "tenant_qa": "tenant_qa",
        "payment": "payment",
        "maintenance": "maintenance",
        "forecast": "forecast",
        "unknown": "unknown",
    },

)

for node in ("tenant_qa", "payment", "maintenance", "forecast", "unknown"):
    builder.add_edge(node, END)

graph = builder.compile()

print("Agent orchestration graph compiled:")
print("Nodes:", list(graph.nodes))

Agent orchestration graph compiled:
Nodes: ['__start__', 'router', 'tenant_qa', 'payment', 'maintenance', 'forecast', 'unknown']


In [9]:
test_cases =[
    {"message": "Can I keep a pet in my unit?"},
    {"message": "Is Bennet Dyani's rent overdue?"},
    {"message": "The kitchen tap is leaking, please schedule a repair."},
    {"message": "What is the projected Q3 rental income?"},
    {"message": "Hello, how are you?"},
]

results = []
for case in test_cases:
    final_state = graph.invoke(case)
    results.append(
        {
            "message":          case["message"],
            "intent":           final_state.get("intent"),
            "confidence":       final_state.get("confidence"),
            "agent_used":       final_state.get("agent_used"),
            "response_preview": (final_state.get("response") or "")[:120],
        }
    )
    pprint(results)

[{'agent_used': 'TenantQAAgent',
  'confidence': 1.0,
  'intent': 'tenant_qa',
  'message': 'Can I keep a pet in my unit?',
  'response_preview': '[Mock LLM] Based on the retrieved context: \n'
                      ' [1] '
                      'C:\\agentic\\Property-Rental-Management-System\\notebooks\\_generated_asset'}]
[{'agent_used': 'TenantQAAgent',
  'confidence': 1.0,
  'intent': 'tenant_qa',
  'message': 'Can I keep a pet in my unit?',
  'response_preview': '[Mock LLM] Based on the retrieved context: \n'
                      ' [1] '
                      'C:\\agentic\\Property-Rental-Management-System\\notebooks\\_generated_asset'},
 {'agent_used': 'PaymentTrackerAgent',
  'confidence': 1.0,
  'intent': 'payment',
  'message': "Is Bennet Dyani's rent overdue?",
  'response_preview': '[Mock LLM] Based on the retrieved context: \n'
                      ' No matching context found.'}]
[{'agent_used': 'TenantQAAgent',
  'confidence': 1.0,
  'intent': 'tenant_qa',
  'message': '

In [13]:
memory = MemorySaver()
persistent_graph = builder.compile(checkpointer=memory)

config = {"configurable": {"thread_id": "session-bennet-A1"}}

turn_1 = persistent_graph.invoke(
    {"message": "What are the quest hours for unit B2?", "tenant_id": "Bennet", "unit_id": "2B"},
    config=config,
)
print("Turn 1 response:", turn_1["response"][:200])

turn_2 = persistent_graph.invoke(
    {"message": "And what about parking?", "tenant_id": "Itumeleng Bedesho", "unit_id": "B2"},
    config=config,
)

print("Turn 2 response:", turn_2["response"] [:200])

history = list(persistent_graph.get_state_history(config))
print(f"\nHistory: {len(history)} snapshots stored in memory.")

Turn 1 response: [Mock LLM] Based on the retrieved context: 
 [1] C:\agentic\Property-Rental-Management-System\notebooks\_generated_assets\06_agent_orchestration\tenant_policy.txt (text)
Unit A1 lease: monthly rent R4
Turn 2 response: [Mock LLM] Based on the retrieved context: 
 [1] C:\agentic\Property-Rental-Management-System\notebooks\_generated_assets\06_agent_orchestration\tenant_policy.txt (text)
Quiet hours 22:00-07:00.Roofto

History: 8 snapshots stored in memory.
